# ComplyAgent: Training Data Extraction

**What this notebook does:** it makes a 10,000-row table from the LendingClub CSV. We use it to train 2 models:
- `hardship_label` (will this customer ask for hardship help?)
- `continued_delinquency_label` (will this customer stay late on payments?)

**Steps in simple words**
1. Read the LendingClub CSV
2. Keep only loans with a clear result
3. Make the 2 labels from real results
4. Make the real features
5. Pick 10,000 rows (balanced)
6. Make the "today" payment columns (simulated)
7. Make fake names, emails, phones
8. Save the CSV, then load it into MySQL

**Which columns are real, simulated, or fake?**

| Type | Columns |
|---|---|
| Real (from LendingClub) | `annual_income`, `loan_amount`, `installment_amount`, `start_date`, `interest_rate`, `loan_term`, `grade`, `dti`, `fico_range_low`, both labels |
| Simulated (we create them) | `days_overdue`, `overdue_amount`, `total_late_payments`, `outstanding_balance`, `total_payment_amount` |
| Fake (Faker library) | `full_name`, `email`, `phone` |

> **Important:** the simulated columns are made **without looking at the labels**. So they cannot leak the answer to the model. They are for MySQL and the Agent demo. The model learns from the **real** columns only (see `MODEL_FEATURES` in step 11).

## 1. Install and import
Load the libraries we need.

In [15]:
%pip install pandas numpy faker scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [16]:
import numpy as np
import pandas as pd
from faker import Faker

## 2. Settings
- `CSV_PATH`: where your LendingClub file is. **Change this to your path.**
- `TOTAL_ROWS`: how many rows we want at the end
- `MAX_HARDSHIP`: max rows with hardship = 1 (they are rare in real data)
- `SEED`: keeps the random results the same every run

In [17]:
CSV_PATH = "../data/raw/accepted_2007_to_2018Q4.csv"
TOTAL_ROWS = 10_000
MAX_HARDSHIP = 1_500      # cap on hardship=1 rows (rare in real data)
SEED = 42

rng = np.random.default_rng(SEED)
fake = Faker()
Faker.seed(SEED)

## 3. Load only the columns we need
The full file has 150+ columns. We load only 15, so it uses less memory.
- **Real features:** `loan_amnt`, `term`, `int_rate`, `installment`, `issue_d`, `annual_inc`, `grade`, `dti`, `fico_range_low`
- **For labels:** `loan_status`, `hardship_flag`, `debt_settlement_flag`
- **Loaded but replaced later:** `total_pymnt`, `total_rec_prncp` (step 9 creates better versions)

In [18]:
cols = [
    "loan_amnt", "term", "int_rate", "installment", "issue_d", "annual_inc",
    "grade", "dti", "fico_range_low",
    "loan_status", "total_pymnt", "total_rec_prncp",
    "hardship_flag", "debt_settlement_flag",
]
df = pd.read_csv(CSV_PATH, usecols=cols, low_memory=False)
print("Raw shape:", df.shape)
df.head()

Raw shape: (2260701, 14)


,loan_amnt,term,int_rate,installment,grade,annual_inc,issue_d,loan_status,dti,fico_range_low,total_pymnt,total_rec_prncp,hardship_flag,debt_settlement_flag
0,3600.0,36 months,13.99,123.03,C,55000.0,Dec-2015,Fully Paid,5.91,675.0,4421.723917,3600.00,N,N
1,24700.0,36 months,11.99,820.28,C,65000.0,Dec-2015,Fully Paid,16.06,715.0,25679.660000,24700.00,N,N
2,20000.0,60 months,10.78,432.66,B,63000.0,Dec-2015,Fully Paid,10.78,695.0,22705.924294,20000.00,N,N
3,35000.0,60 months,14.85,829.90,C,110000.0,Dec-2015,Current,17.06,785.0,31464.010000,19102.35,N,N
4,10400.0,60 months,22.45,289.91,F,104433.0,Dec-2015,Fully Paid,25.37,695.0,11740.500000,10400.00,N,N


## 4. Look at loan results
See which results exist and how many loans have each one.

In [19]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

## 5. Keep only clear results
- **Delinquent = 1:** `Charged Off`, `Default`, `Late (31-120 days)`
- **Not delinquent = 0:** `Fully Paid`

**Why not `Current`?** A current loan has no final result yet. It can still go bad later, so calling it 0 would be wrong. Same for `In Grace Period` and `Late (16-30 days)`.

We also drop rows that have missing values in the main columns.

In [20]:
positive = ["Charged Off", "Default", "Late (31-120 days)"]
negative = ["Fully Paid"]

df = df[df["loan_status"].isin(positive + negative)].dropna(
    subset=["loan_amnt", "annual_inc", "installment", "int_rate", "term",
            "issue_d", "grade", "dti", "fico_range_low"]
).copy()
print("After filtering:", df.shape)

After filtering: (1366419, 14)


## 6. Make the 2 labels
Both labels come from **real** LendingClub results.
- `continued_delinquency_label` = 1 if the loan ended as Charged Off, Default, or Late 31-120. Otherwise 0.
- `hardship_label` = 1 if `hardship_flag` is `Y` **or** `debt_settlement_flag` is `Y`. Otherwise 0.

The last cell shows how much the two labels overlap. Settled debts are often also delinquent, so some overlap is normal.

In [28]:
df["continued_delinquency_label"] = df["loan_status"].isin(positive).astype(int)
df["hardship_label"] = (
    (df["hardship_flag"] == "Y") | (df["debt_settlement_flag"] == "Y")
).astype(int)

print(df["continued_delinquency_label"].value_counts())
print(df["hardship_label"].value_counts())
print()
print(pd.crosstab(df["hardship_label"], df["continued_delinquency_label"]))

continued_delinquency_label
0    1076448
1     289971
Name: count, dtype: int64
hardship_label
0    1331507
1      34912
Name: count, dtype: int64

continued_delinquency_label        0       1
hardship_label                              
0                            1076443  255064
1                                  5   34907


## 7. Make the real features
We rename the LendingClub columns to our names. These are all **real data**.

| Our name | From | Notes |
|---|---|---|
| `annual_income` | `annual_inc` | |
| `loan_amount` | `loan_amnt` | |
| `installment_amount` | `installment` | monthly payment |
| `start_date` | `issue_d` | turned into a date |
| `interest_rate` | `int_rate` | |
| `loan_term` | `term` | 36 or 60 months |
| `grade_num` | `grade` | A=1 (safest) to G=7 (riskiest) |
| `dti` | `dti` | debt-to-income % |
| `fico_range_low` | `fico_range_low` | credit score |
| `loan_to_income` | `loan_amount / annual_income` | how big the loan is for the income |

In [29]:
df["annual_income"] = df["annual_inc"]
df["loan_amount"] = df["loan_amnt"]
df["installment_amount"] = df["installment"]
df["start_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y").dt.date
df["interest_rate"] = df["int_rate"]
df["loan_term"] = df["term"].str.extract(r"(\d+)").astype(int)
df["grade_num"] = df["grade"].map({g: i + 1 for i, g in enumerate("ABCDEFG")})
df["loan_to_income"] = (df["loan_amnt"] / df["annual_inc"].clip(lower=1)).round(4)

## 8. Pick 10,000 balanced rows
- Take up to 1,500 rows with hardship = 1.
- Fill the rest with hardship = 0 rows.
- Make `continued_delinquency` exactly **5,000 / 5,000**.

**Careful:** real life is not 50/50. So the probabilities from the models will look too high. We must **calibrate** them before the Agent uses them. For the hardship model use `class_weight="balanced"`.

In [30]:
hard = df[df["hardship_label"] == 1]
h = hard.sample(n=min(len(hard), MAX_HARDSHIP), random_state=SEED)
rest = df.drop(h.index)

half = TOTAL_ROWS // 2
need1 = max(half - int(h["continued_delinquency_label"].sum()), 0)
need0 = max(half - int((h["continued_delinquency_label"] == 0).sum()), 0)

rest_nohard = rest[rest["hardship_label"] == 0]
pos = rest_nohard[rest_nohard["continued_delinquency_label"] == 1].sample(need1, random_state=SEED)
neg = rest_nohard[rest_nohard["continued_delinquency_label"] == 0].sample(need0, random_state=SEED)

data = pd.concat([h, pos, neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Rows:", len(data))
print(data["continued_delinquency_label"].value_counts())
print(data["hardship_label"].value_counts())

Rows: 10000
continued_delinquency_label
0    5000
1    5000
Name: count, dtype: int64
hardship_label
0    8500
1    1500
Name: count, dtype: int64


## 9. Make the "today" payment columns (simulated)
LendingClub has no daily payment history, so we create these columns ourselves.

**Main rule: we do NOT use the labels here.** Every customer gets numbers from the same random rules. So these columns cannot leak the answer.

How each column is made:
- `days_overdue`: random. About 45% are 1-30 days, 30% are 31-60, 15% are 61-90, 10% are 91-120.
- `overdue_amount`: monthly payment x number of months overdue
- `k`: how many months into the loan the customer is today (random)
- `outstanding_balance`: normal loan formula, balance left after the payments made so far
- `total_payment_amount`: monthly payment x payments made so far
- `total_late_payments`: months overdue + a small random number, never more than `k`

Because they are random, they will **not** help the model predict anything. That is on purpose. They are for the database and the Agent demo.

In [24]:
n = len(data)

# 1) Days overdue today: same random rule for everyone (labels are NOT used)
bucket = rng.choice(4, size=n, p=[0.45, 0.30, 0.15, 0.10])
low  = np.array([1, 31, 61, 91])
high = np.array([30, 60, 90, 120])
days = rng.integers(low[bucket], high[bucket] + 1)
months_overdue = np.ceil(days / 30).astype(int)

data["days_overdue"] = days
data["overdue_amount"] = (data["installment_amount"] * months_overdue).round(2)

# 2) Snapshot month k = how far into the loan the customer is today
term = data["loan_term"].to_numpy()
k = rng.integers(6, term - 1, n)
kp = np.maximum(k - months_overdue, 1)      # installments actually paid

# 3) Balance left after kp payments (standard loan formula)
r = data["interest_rate"].to_numpy() / 1200
P = data["loan_amount"].to_numpy()
bal = P * ((1 + r) ** term - (1 + r) ** kp) / ((1 + r) ** term - 1)

data["outstanding_balance"]  = bal.round(2)
data["total_payment_amount"] = (data["installment_amount"] * kp).round(2)

# 4) Late payments so far (labels are NOT used)
data["total_late_payments"] = np.minimum(
    months_overdue + rng.poisson(0.5, n), k
)

## 10. Make fake customer details
LendingClub hides real names, emails, and phones. So we make fake ones with Faker.

These are only for MySQL. **Never use them as model features.**

In [25]:
data["full_name"] = [fake.name() for _ in range(n)]
data["email"] = [fake.unique.email() for _ in range(n)]
data["phone"] = [fake.phone_number() for _ in range(n)]

## 11. Final table and save
- `final_cols`: all columns saved to the CSV. The first 16 are your MySQL plan. The last 4 are extra real features for training.
- `MODEL_FEATURES`: the columns the models should learn from. Only **real** columns are here.
- `customer_id`, `loan_id`, `payment_id` are not saved, because MySQL makes them with `AUTO_INCREMENT`.

**Why is `start_date` not in `MODEL_FEATURES`?** `hardship_flag` only exists for newer loans. If the model sees the year, it can just guess "new loan = hardship". That is not real learning.

In [26]:
final_cols = [
    "full_name", "annual_income", "email", "phone",
    "loan_amount", "installment_amount", "start_date", "outstanding_balance",
    "overdue_amount", "days_overdue",
    "total_late_payments", "total_payment_amount",
    "interest_rate", "loan_term",
    "hardship_label", "continued_delinquency_label",
    # extra real features for training
    "grade_num", "dti", "fico_range_low", "loan_to_income",
]
out = data[final_cols]
out.to_csv("../data/processed/complyagent_training_10k.csv", index=False)

# Columns the models should learn from (real data only)
MODEL_FEATURES = [
    "annual_income", "loan_amount", "installment_amount", "interest_rate",
    "loan_term", "grade_num", "dti", "fico_range_low", "loan_to_income",
]
out.head()

,full_name,annual_income,email,phone,loan_amount,installment_amount,start_date,outstanding_balance,overdue_amount,days_overdue,total_late_payments,total_payment_amount,interest_rate,loan_term,hardship_label,continued_delinquency_label,grade_num,dti,fico_range_low,loan_to_income
0,Allison Hill,56000.0,silvapreston@example.org,444-615-6574x3759,15000.0,385.33,2014-12-01,12429.82,1155.99,62,5,5779.95,18.54,60,0,0,5,4.37,680.0,0.2679
1,Noah Rhodes,160000.0,kathleenellis@example.com,+1-413-728-1801,35000.0,1179.12,2015-02-01,22042.66,1179.12,22,1,17686.80,12.99,36,0,1,3,22.10,660.0,0.2188
2,Angie Henderson,41265.0,jeffrey20@example.com,(984)345-7352x640,13125.0,407.61,2016-06-01,3558.00,1222.83,67,3,11005.47,7.39,36,0,1,1,21.58,745.0,0.3181
3,Daniel Wagner,45000.0,cbradley@example.org,001-649-213-1046x527,9000.0,320.83,2016-12-01,7797.71,641.66,52,2,1924.98,16.99,36,0,1,4,30.85,660.0,0.2000
4,Cristian Santos,57500.0,rhonda97@example.org,001-445-864-5431x2903,19450.0,468.97,2015-07-01,17159.57,468.97,17,2,4689.70,15.61,60,0,1,4,30.66,715.0,0.3383


## 12. Sanity checks
**Check 1: basic shape.** 10,000 rows, no nulls, and the label counts.

**Check 2: leakage test.** For each column we compute an **AUC** score against each label.
- **0.5** = the column tells us nothing
- **1.0** = the column gives away the answer

What you should see:
- Simulated columns (`days_overdue` etc.): close to **0.5**. Good, no leakage.
- Real columns (`interest_rate`, `fico_range_low`, `grade_num`): a bit above 0.5, around 0.55-0.70. This is real, honest signal.

**Check 3: hardship by year.** If the hardship rate changes a lot by year, `start_date` would be a shortcut for the model.

In [27]:
from sklearn.metrics import roc_auc_score

# Check 1: shape and labels
print("Shape:", out.shape)
print("Nulls:\n", out.isnull().sum()[out.isnull().sum() > 0])
print("\nContinued delinquency:\n", out["continued_delinquency_label"].value_counts())
print("\nHardship:\n", out["hardship_label"].value_counts())

# Check 2: leakage test (AUC per column; strength = distance from 0.5)
sim_cols = ["days_overdue", "overdue_amount", "total_late_payments",
            "outstanding_balance", "total_payment_amount"]
rows = []
for c in sim_cols + MODEL_FEATURES:
    row = {"column": c, "type": "simulated" if c in sim_cols else "real"}
    for lab_name in ["continued_delinquency_label", "hardship_label"]:
        auc = roc_auc_score(out[lab_name], out[c])
        row[lab_name] = round(max(auc, 1 - auc), 3)
    rows.append(row)
print("\nAUC per column (0.5 = no signal):")
print(pd.DataFrame(rows).to_string(index=False))

# Check 3: hardship rate by start year
year = pd.to_datetime(out["start_date"]).dt.year
print("\nHardship rate by start year:")
print(out.groupby(year)["hardship_label"].mean().round(3))

Shape: (10000, 20)
Nulls:
 Series([], dtype: int64)

Continued delinquency:
 continued_delinquency_label
0    5000
1    5000
Name: count, dtype: int64

Hardship:
 hardship_label
0    8500
1    1500
Name: count, dtype: int64

AUC per column (0.5 = no signal):
              column      type  continued_delinquency_label  hardship_label
        days_overdue simulated                        0.506           0.504
      overdue_amount simulated                        0.534           0.522
 total_late_payments simulated                        0.502           0.500
 outstanding_balance simulated                        0.555           0.530
total_payment_amount simulated                        0.559           0.540
       annual_income      real                        0.542           0.521
         loan_amount      real                        0.560           0.537
  installment_amount      real                        0.550           0.530
       interest_rate      real                        0.6

## Next steps
1. Load `complyagent_training_10k.csv` into MySQL.
2. Train the 2 models using only `MODEL_FEATURES`.
3. **Calibrate** the probabilities (the data is balanced, real life is not).
4. Save `prob_hardship` and `prob_continued_delinquency` into the `risk` table.